<a href="https://colab.research.google.com/github/truegeo/python-async-programming/blob/main/Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercise 1: The Fintech Dashboard API Aggregator

**The Scenario:**
You are building the backend for a financial dashboard. The user requests the current price of Bitcoin, Apple stock, and the GBP/USD exchange rate. These require calling three separate, slow third-party APIs.
If you fetch them sequentially (synchronously), the user waits for the sum of all response times. If you fetch them concurrently (asynchronously), the user only waits for the slowest individual API.

**Your Task:**
1. **The Sync Baseline:** Write three synchronous functions (`fetch_crypto_sync`, `fetch_stock_sync`, `fetch_forex_sync`) that use `time.sleep()` to simulate network delays of 1s, 2s, and 1.5s respectively. Run them sequentially and time the total execution.
2. **The Async Implementation:** Write three asynchronous equivalents (`fetch_crypto`, `fetch_stock`, `fetch_forex`) using `asyncio.sleep()`.
3. **The Concurrency Engine:** Create an async main function. Use `asyncio.gather()` to trigger all three async functions at the exact same time. Time the total execution.

**The Senior Engineer Challenge:**
APIs are notoriously unreliable. Hardcode your asynchronous `fetch_stock` function to raise an `Exception("Stock API Down")`.
Modify your `asyncio.gather()` call so that a failure in one API does *not* crash the whole application, but instead safely returns the Exception object alongside the successful data.

In [2]:
import asyncio
import time

# ==========================================
# 1. SYNCHRONOUS IMPLEMENTATION (BLOCKING)
# ==========================================
def fetch_crypto_sync():
    time.sleep(1)
    return 100

def fetch_stock_sync():
    time.sleep(2)
    return 150

def fetch_forex_sync():
    time.sleep(1.5)
    return 200

def fetch_all_sync():
    start = time.time()
    crypto = fetch_crypto_sync()
    stock = fetch_stock_sync()
    forex = fetch_forex_sync()
    end = time.time()
    print(f"Sync Time taken: {end - start:.2f} seconds")
    return [crypto, stock, forex]

# ==========================================
# 2. ASYNCHRONOUS IMPLEMENTATION (NON-BLOCKING)
# ==========================================
async def fetch_crypto():
    await asyncio.sleep(1)
    return 100

async def fetch_stock():
    await asyncio.sleep(2)
    raise Exception("Stock API Down") # Simulating an API failure

async def fetch_forex():
    await asyncio.sleep(1.5)
    return 200

async def fetch_all_async():
    start = time.time()
    # return_exceptions=True prevents the Exception in fetch_stock from crashing the gather call
    output = await asyncio.gather(
        fetch_crypto(),
        fetch_stock(),
        fetch_forex(),
        return_exceptions=True
    )
    end = time.time()
    print(f"Async Time taken: {end - start:.2f} seconds")
    return output

# ==========================================
# 3. EXECUTION
# ==========================================
print("-------------------------------- START --------------------------------")
print(fetch_all_sync())
print("-------------------------------- SYNC FINISH --------------------------------\n")

print("-------------------------------- ASYNC STARTING --------------------------------")
# Note: Top-level await works perfectly in Colab/Jupyter environments
print(await fetch_all_async())
print("-------------------------------- ASYNC FINISH --------------------------------")

-------------------------------- START --------------------------------
Sync Time taken: 4.50 seconds
[100, 150, 200]
-------------------------------- SYNC FINISH --------------------------------

-------------------------------- ASYNC STARTING --------------------------------
Async Time taken: 2.00 seconds
[100, Exception('Stock API Down'), 200]
-------------------------------- ASYNC FINISH --------------------------------


### Exercise 1 Outcomes & Takeaways

**1. Sequential vs. Concurrent Execution**
- **Synchronous Time:** ~4.5 seconds. The program halts entirely at each `time.sleep()`. It cannot start the Stock API call until the Crypto call finishes.
- **Asynchronous Time:** ~2.0 seconds. When `fetch_crypto()` hits `asyncio.sleep()`, it yields control back to the Event Loop, which immediately kicks off `fetch_stock()` and `fetch_forex()`. The total time is simply the duration of the single longest task (2 seconds).

**2. Fault Tolerance in Production**
By setting `return_exceptions=True` in `asyncio.gather()`, we prevented the entire application from failing due to one bad microservice.
The resulting array `[100, Exception('Stock API Down'), 200]` gives us exactly what we need: we can render the Crypto and Forex widgets on the frontend while showing a graceful "Data Unavailable" error on the Stock widget.

**Real-world application:** This is the exact pattern used in modern Python web frameworks like **FastAPI** to handle high-throughput I/O-bound requests.

---
---

## Exercise 2: The "Polite" Data Pipeline (Rate Limiting)

**The Scenario:**
You are a Data Engineer tasked with pulling 500 user profiles from a government database API.
Because it is a public API, it has a strict rate limit: **If you make more than 10 concurrent requests, your IP address will be permanently banned.**

If you run this synchronously, it takes a long time. If you run this using a basic `asyncio.gather()` (like in Exercise 1), Python will instantly fire off 500 requests at the exact same millisecond, and you will get banned. You need to throttle your async code.

**Your Task:**
1. **The Mock API:** Write an async function called `download_profile(profile_id: int)`. It should simulate network latency by sleeping for 0.5 seconds, then return a string like `"Profile {profile_id} downloaded"`.
2. **The Semaphore:** In your main function, instantiate an `asyncio.Semaphore` with a limit of 10.
3. **The Limiter:** Modify your `download_profile` logic (or create a wrapper function) so that the actual "download" (the sleep) is protected by the semaphore using an `async with` context manager.
4. **The Execution:** Create a list of 500 tasks and run them all concurrently using `asyncio.gather()`.
5. **The Benchmark:** Time the total execution. (Hint: 500 downloads, processed 10 at a time, taking 0.5s per batch, should take exactly ~25 seconds to complete).

**The Senior Engineer Challenge (Optional but Recommended):**
Network requests fail. Modify your `download_profile` function to randomly raise a `ConnectionError` about 10% of the time (use the `random` module).
Catch this error locally and implement a basic "Retry" loop inside the function. If it fails, print "Retrying...", wait 1 second, and try again. Ensure it doesn't try more than 3 times before finally giving up.

In [6]:
import random

async def download_profile(profile_id: int) -> str:
  if random.random() < 0.1:
    raise ConnectionError("Network Error")
  await asyncio.sleep(0.5)
  return f"Profile {profile_id} downloaded"

async def wrapper(profile_id: int, sem: asyncio.Semaphore) -> str:
  async with sem:
    max_retries = 3
    for attempt in range(max_retries):
      try:
        return await download_profile(profile_id)
      except ConnectionError:
        if attempt < max_retries:
          print(f"Retrying... (Attempt {attempt + 1}/{max_retries})")
          # "Polite" backoff: wait a bit before retrying
          await asyncio.sleep(1)
  # If the loop finishes all 3 attempts without returning success:
  return f"Profile {profile_id} failed completely after {max_retries} retries"



async def main():
  start = time.time()
  sem = asyncio.Semaphore(10)
  tasks = [wrapper(i, sem) for i in range(500)]
  async_results = await asyncio.gather(*tasks)
  end = time.time()
  print(f"Time taken: {end - start:.2f} seconds")
  return async_results

await main()




Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 2/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 2/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... (Attempt 1/3)
Retrying... 

['Profile 0 downloaded',
 'Profile 1 downloaded',
 'Profile 2 downloaded',
 'Profile 3 downloaded',
 'Profile 4 downloaded',
 'Profile 5 downloaded',
 'Profile 6 downloaded',
 'Profile 7 downloaded',
 'Profile 8 downloaded',
 'Profile 9 downloaded',
 'Profile 10 downloaded',
 'Profile 11 downloaded',
 'Profile 12 downloaded',
 'Profile 13 downloaded',
 'Profile 14 downloaded',
 'Profile 15 downloaded',
 'Profile 16 downloaded',
 'Profile 17 downloaded',
 'Profile 18 downloaded',
 'Profile 19 downloaded',
 'Profile 20 downloaded',
 'Profile 21 downloaded',
 'Profile 22 downloaded',
 'Profile 23 downloaded',
 'Profile 24 downloaded',
 'Profile 25 downloaded',
 'Profile 26 downloaded',
 'Profile 27 downloaded',
 'Profile 28 downloaded',
 'Profile 29 downloaded',
 'Profile 30 downloaded',
 'Profile 31 downloaded',
 'Profile 32 downloaded',
 'Profile 33 downloaded',
 'Profile 34 downloaded',
 'Profile 35 downloaded',
 'Profile 36 downloaded',
 'Profile 37 downloaded',
 'Profile 38 downloade

### Exercise 2 Outcomes & Takeaways

**1. Concurrency Control with Semaphores**
Without a Semaphore, `asyncio.gather()` would have fired all 500 requests into the event loop at the exact same time, likely crashing the server, getting our IP banned, or exhausting our local connection pool.
By wrapping the execution in `async with asyncio.Semaphore(10)`, we created a "toll booth" that perfectly regulated the flow to 10 active tasks at a time. This resulted in a controlled execution time of ~25-35 seconds.

**2. The Retry with Backoff Pattern**
Network I/O is inherently flaky. By anticipating `ConnectionError`, we prevented the task from failing immediately. Using a `for` loop combined with `asyncio.sleep()`, we created an asynchronous backoff.
*Crucially*, while one task was sleeping during its retry, it did *not* block the Event Loop—though it *did* hold onto its Semaphore token, ensuring we never accidentally exceeded our strict rate limit of 10 during the retries.

**Real-world application:**
This is an absolute necessity for modern Data Engineering and Backend pipelines. Whether scraping public APIs, ingesting bulk data into AWS S3, or talking to database connection pools (like `asyncpg` with PostgreSQL), resource throttling via Semaphores is a required skill for senior roles.

---
---

## Exercise 3: The High-Throughput Event Processor (Producer-Consumer)

**The Scenario:**
You are building the analytics backend for a high-traffic e-commerce platform. Every time a user clicks a button, the web server receives an event. You must save these events to a database.
If the web server waits for the database to write the event before responding to the user, the website will feel incredibly slow.

Instead, the web server should act as a **Producer**: it instantly drops the event into a holding area (a Queue) and returns a success response to the user.
Meanwhile, a pool of background workers (**Consumers**) constantly monitor the Queue, pick up the events, and write them to the database at their own pace.

**Your Task:**
1. **The Queue:** In your main function, create an `asyncio.Queue()`.
2. **The Producer:** Write an async function that accepts the queue. Have it generate 20 "events" (a simple string or dictionary) using a `for` loop. For each event, print "Produced [event]", put it into the queue using `await queue.put()`, and sleep for `0.1` seconds to simulate incoming web traffic.
3. **The Consumer:** Write an async worker function that accepts the queue and a `worker_id`. It should run an infinite `while True:` loop. Inside the loop:
   - Wait for an item: `event = await queue.get()`
   - Simulate a slow DB write: `await asyncio.sleep(0.5)`
   - Print a success message: "Worker {id} saved {event}"
   - Crucially, tell the queue the task is done: `queue.task_done()`
4. **The Orchestrator:** In `main()`, use `asyncio.create_task()` to start the Producer and 3 Consumers simultaneously in the background.

**The Senior Engineer Challenge (Graceful Shutdown):**
If you just run the tasks, your Colab cell will never stop executing because the Consumers are stuck in an infinite `while True:` loop!
To shut down cleanly:
1. `await` your Producer task to finish generating its 20 events.
2. `await queue.join()` — This is the magic command. It pauses your main script until the queue is completely empty and every `task_done()` has been called.
3. Iterate through your Consumer tasks and call `task.cancel()` on them so the Colab cell can successfully finish.

In [5]:
async def producer(queue: asyncio.Queue):
  for i in range(20):
    event = f"Event {i}"
    print(f"Produced {event}")
    await queue.put(event)
    await asyncio.sleep(0.1)

async def consumer(queue: asyncio.Queue, worker_id: int):
  while True:
    event = await queue.get()
    await asyncio.sleep(0.5)
    print(f"Worker {worker_id} saved {event}")
    queue.task_done()

async def main():
  queue = asyncio.Queue()
  producer_task = asyncio.create_task(producer(queue))
  consumer_tasks = [asyncio.create_task(consumer(queue, i)) for i in range(3)]
  await producer_task
  await queue.join()
  for task in consumer_tasks:
    task.cancel()

await main()

Produced Event 0
Produced Event 1
Produced Event 2
Produced Event 3
Produced Event 4
Worker 0 saved Event 0
Produced Event 5
Worker 1 saved Event 1
Produced Event 6
Worker 2 saved Event 2
Produced Event 7
Produced Event 8
Produced Event 9
Worker 0 saved Event 3
Produced Event 10
Worker 1 saved Event 4
Produced Event 11
Worker 2 saved Event 5
Produced Event 12
Produced Event 13
Produced Event 14
Worker 0 saved Event 6
Produced Event 15
Worker 1 saved Event 7
Produced Event 16
Worker 2 saved Event 8
Produced Event 17
Produced Event 18
Produced Event 19
Worker 0 saved Event 9
Worker 1 saved Event 10
Worker 2 saved Event 11
Worker 0 saved Event 12
Worker 1 saved Event 13
Worker 2 saved Event 14
Worker 0 saved Event 15
Worker 1 saved Event 16
Worker 2 saved Event 17
Worker 0 saved Event 18
Worker 1 saved Event 19


### Exercise 3 Outcomes & Takeaways

**1. Decoupling with Producer-Consumer**
We successfully decoupled the ingestion of data (the Producer) from the processing of data (the Consumers). In a high-throughput backend, this pattern is how we prevent slow database writes from holding up fast API responses. The `asyncio.Queue` safely passes data between these concurrent tasks without memory leaks or race conditions.

**2. Managing Background Workers**
Using `asyncio.create_task()` allows us to spawn background workers that live outside our main procedural code flow. Because they run on an infinite `while True:` loop, they act as persistent daemon processes constantly listening for new work.

**3. Graceful Shutdown**
In production, you cannot abruptly kill a server during deployment if it has unprocessed events in its queue. By coordinating `await queue.join()` (which waits for all `task_done()` signals) and then explicitly calling `task.cancel()` on our workers, we ensure zero data loss during a system shutdown.

---
---

## Exercise 4: The Unreliable Microservice Caller (Timeouts & Cancellation)

**The Scenario:**
You are building the checkout process for a payment gateway. It talks to a legacy banking system to process credit cards. Usually, it takes 1 second. However, sometimes the banking API hangs indefinitely.
If the API hangs for more than 3 seconds, you must cancel the request, trigger a fallback system, and ensure that any open database connections are safely closed despite the cancellation.

**Your Task:**
1. **The Unreliable API:** Write an async function `process_payment()`. Inside it, simulate a system that *sometimes* hangs. Use `random.choice([1, 10])` to either sleep for 1 second (success) or 10 seconds (hang). Return `"Payment Successful"`.
2. **The Timeout:** Write a `checkout()` function. Use `asyncio.wait_for()` to call `process_payment()`, but enforce a strict timeout of `3.0` seconds.
3. **The Fallback:** Wrap the `wait_for` in a `try/except` block targeting `TimeoutError` (or `asyncio.TimeoutError`). If it times out, catch the error and return `"Fallback Payment Executed"`. Execute `checkout()` in your main block a few times to see both outcomes.

**The Senior Engineer Challenge (The `finally` block):**
When `asyncio.wait_for` times out, it doesn't just stop listening—it aggressively throws an `asyncio.CancelledError` *inside* the running `process_payment()` task to kill it.
Modify your `process_payment()` function to use a `try/finally` block. Inside the `finally` block, print `"Cleaning up database connections..."`.
Observe the logs to prove that even when the task is abruptly cancelled by the timeout, the cleanup code is guaranteed to run.

In [28]:
async def process_payment():
  try:
    random_delay = random.choice([1,10])
    await asyncio.sleep(random_delay)
    return "Payment Successful"
  finally:
    print("Cleaning up database connections")

async def checkout():
  task = asyncio.create_task(process_payment())
  try:
    return await asyncio.wait_for(task, timeout=3)
  except TimeoutError:
    return "Fallback Payment Executed"

await checkout()

Cleaning up database connections


'Fallback Payment Executed'

### Exercise 4 Outcomes & Takeaways

**1. Protecting Systems with Timeouts**
In distributed architectures, you cannot trust third-party microservices to always respond promptly. By wrapping an unreliable task in `asyncio.wait_for(..., timeout=3.0)`, we guarantee that our own web application will never hang indefinitely. This ensures a smooth User Experience by allowing us to trigger a fallback system (like a secondary payment provider or a graceful error message).

**2. The Anatomy of Task Cancellation**
When `wait_for` hits its timeout limit, it injects an `asyncio.CancelledError` directly into the running task. The task raises this error exactly at the `await` statement where it was currently paused.

**3. Bulletproof Resource Cleanup**
Because tasks can be violently cancelled from the outside, we must protect our resources (like database connections or open files). By wrapping the `await` operations in a `try...finally` block, we guarantee that our cleanup code will execute regardless of whether the task finishes naturally or is forcefully cancelled by a timeout. This prevents memory leaks and locked database tables in production.